In [ ]:
# setup for Python 3.13
from mlwpy import *

digits = datasets.load_digits()
digits_ftrs, digits_tgt = digits.data, digits.target

diabetes = datasets.load_diabetes()
diabetes_ftrs, diabetes_tgt = diabetes.data, diabetes.target

iris = datasets.load_iris()
tts = skms.train_test_split(
    iris.data,
    iris.target,
    test_size=0.75,
    stratify=iris.target
)

(iris_train_ftrs, iris_test_ftrs,
 iris_train_tgt, iris_test_tgt) = tts


In [ ]:
base_estimators = [
    linear_model.LogisticRegression(max_iter=1000),
    tree.DecisionTreeClassifier(max_depth=3),
    naive_bayes.GaussianNB()
]

base_estimators = [(get_model_name(m), m) for m in base_estimators]

ensemble_model = ensemble.VotingClassifier(estimators=base_estimators)

cv_scores = skms.cross_val_score(ensemble_model, digits_ftrs, digits_tgt)

print(cv_scores)


In [ ]:
dataset = np.array([1,5,10,10,17,20,35])
def compute_mean(data):
    return np.sum(data) / data.size
compute_mean(dataset)

In [ ]:
def bootstrap_sample(data):
    n = len(data)
    idx = np.arange(n)
    bs_idx = np.random.choice(idx, n, replace=True)  # default added for clarity
    return data[bs_idx]


In [ ]:
bsms = []

for i in range(5):
    bs_sample = bootstrap_sample(dataset)
    bs_mean = compute_mean(bs_sample)
    bsms.append(bs_mean)
    print(bs_sample, "{:5.2f}".format(bs_mean))


In [ ]:
print("{:5.2f}".format(sum(bsms) / len(bsms)))

In [ ]:
def compute_bootstrap_statistic(data, num_boots, statistic):
    bs_stats = []
    #Repeatedly calculate statistic on num_boots bootstrap samples
    for i in range(num_boots):
        sample = bootstrap_sample(data)
        stat = statistic(sample)
        bs_stats.append(stat)
    return np.mean(bs_stats)  # clearer than sum()/num_boots

bs_mean = compute_bootstrap_statistic(dataset, 100, compute_mean)
print("{:5.2f}".format(bs_mean))


In [ ]:
def make_knn_statistic(new_example):
    def knn_statistic(dataset):
        ftrs, tgt = dataset[:, :-1], dataset[:, -1]
        knn = neighbors.KNeighborsRegressor(n_neighbors=3).fit(ftrs, tgt)
        return knn.predict(new_example)
    return knn_statistic


In [ ]:
# have to slightly massage data for this scenario
# we use last example as our fixed test example
diabetes_dataset = np.c_[diabetes_ftrs, diabetes_tgt]

ks = make_knn_statistic(diabetes_ftrs[-1].reshape(1, -1))

compute_bootstrap_statistic(diabetes_dataset, 100, ks)


In [ ]:
def bagged_learner(dataset, base_model, num_models=10):
    # Convert dataset to features and targets
    x, y = dataset[:, :-1], dataset[:, -1]
    n = len(dataset)
    models = []
    for _ in range(num_models):
        bs_idx = np.random.choice(np.arange(n), n, replace=True)
        x_bs, y_bs = x[bs_idx], y[bs_idx]
        model = base_model().fit(x_bs, y_bs)
        models.append(model)
    return models

def bagged_predict_class(models, example):
    # Take the most frequent (mode) predicted class as result
    preds = [m.predict(example)[0] for m in models]
    return pd.Series(preds).mode()[0]


In [ ]:
def my_boosted_classifier(base_classifier, bc_args, examples, targets, m):
    n = len(examples)
    data_weights = np.full(n, 1 / n)
    models, model_weights = [], []

    for i in range(m):
        weighted_dataset = reweight((examples, targets), data_weights)
        this_model = base_classifier(*bc_args).fit(*weighted_dataset)

        errors = this_model.predict(examples) != targets
        weighted_error = np.dot(data_weights, errors)

        # magic reweighting steps
        this_model_wgt = np.log((1 - weighted_error) / weighted_error)
        data_weights *= np.exp(this_model_wgt * errors)
        data_weights /= data_weights.sum()  # normalize to 1.0

        models.append(this_model)
        model_weights.append(this_model_wgt)

    return ensemble.VotingClassifier(
        estimators=[(f"model_{i}", m) for i, m in enumerate(models)],
        voting='soft',
        weights=model_weights
    )


In [ ]:
model = ensemble.AdaBoostClassifier()

stage_preds = (
    model.fit(iris_train_ftrs, iris_train_tgt)
          .staged_predict(iris_test_ftrs)
)

stage_scores = [
    metrics.accuracy_score(iris_test_tgt, pred)
    for pred in stage_preds
]

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
ax.plot(stage_scores)
ax.set_xlabel('# steps')
ax.set_ylabel('accuracy')


In [ ]:
def fit_predict_score(model, ds):
    return skms.cross_val_score(model, *ds, cv=10).mean()

stump = tree.DecisionTreeClassifier(max_depth=1)
dtree = tree.DecisionTreeClassifier(max_depth=3)
forest = ensemble.RandomForestClassifier(max_features=1, max_depth=1)

tree_classifiers = {
    'stump': stump,
    'dtree': dtree,
    'forest': forest
}

max_est = 100
data = (digits_ftrs, digits_tgt)

stump_score = fit_predict_score(stump, data)
tree_score = fit_predict_score(dtree, data)

forest_scores = [
    fit_predict_score(forest.set_params(n_estimators=n), data)
    for n in range(1, max_est + 1)
]


In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))

xs = list(range(1, max_est + 1))

ax.plot(xs, np.repeat(stump_score, max_est), label='stump')
ax.plot(xs, np.repeat(tree_score, max_est), label='tree')
ax.plot(xs, forest_scores, label='forest')

ax.set_xlabel('Number of Trees in Forest')
ax.set_ylabel('Accuracy')
ax.legend(loc='lower right')


In [ ]:
def my_manual_cv(dataset, k=10):
    """manually generate cv-folds from dataset"""
    # expect ftrs, tgt tuple
    ds_ftrs, ds_tgt = dataset
    manual_cv = skms.StratifiedKFold(k).split(ds_ftrs, ds_tgt)

    for train_idx, test_idx in manual_cv:
        train_ftrs = ds_ftrs[train_idx]
        test_ftrs = ds_ftrs[test_idx]
        train_tgt = ds_tgt[train_idx]
        test_tgt = ds_tgt[test_idx]
        yield train_ftrs, test_ftrs, train_tgt, test_tgt


In [ ]:
AdaBC = ensemble.AdaBoostClassifier
GradBC = ensemble.GradientBoostingClassifier

boosted_classifiers = {
    'boost(Ada)': AdaBC(learning_rate=2.0),
    'boost(Grad)': GradBC(loss="log_loss")  # 'deviance' deprecated
}

mean_accs = {}

for name, model in boosted_classifiers.items():
    model.set_params(n_estimators=max_est)
    accs = []

    for tts in my_manual_cv((digits_ftrs, digits_tgt)):
        train_f, test_f, train_t, test_t = tts
        s_preds = model.fit(train_f, train_t).staged_predict(test_f)
        s_scores = [metrics.accuracy_score(test_t, p) for p in s_preds]
        accs.append(s_scores)

    mean_accs[name] = np.array(accs).mean(axis=0)

mean_acc_df = pd.DataFrame.from_dict(mean_accs, orient='columns')


In [ ]:
xs = list(range(1, max_est + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3), sharey=True)

# Left plot: Forest vs. Tree vs. Stump
ax1.plot(xs, np.repeat(stump_score, max_est), label='stump')
ax1.plot(xs, np.repeat(tree_score, max_est), label='tree')
ax1.plot(xs, forest_scores, label='forest')
ax1.set_ylabel('Accuracy')
ax1.set_xlabel('Number of Trees in Forest')
ax1.legend()

# Right plot: Boosting curves
mean_acc_df.plot(ax=ax2)
ax2.set_ylim(0.0, 1.1)
ax2.set_xlabel('# Iterations')
ax2.legend(ncol=2)

plt.tight_layout()
plt.show()


In [ ]:
# conda install py-xgboost
import xgboost

# gives us xgboost.XGBRegressor, xgboost.XGBClassifier
# which interface nicely with sklearn
# see docs at:
# https://xgboost.readthedocs.io/en/latest/parameter.html

xgbooster = xgboost.XGBClassifier(objective="multi:softmax")

scores = skms.cross_val_score(xgbooster, iris.data, iris.target, cv=10)

print(scores)
